# Tumor Segmentation with Manual Annotations from IDC

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial demonstrates how to find manual tumor annotations in IDC and use them to train a tumor segmentation model with MONAI.

## What You'll Learn

1. Discover image-derived DICOM objects (SEG, RTSTRUCT, SR) in IDC
2. Find manual tumor annotations using IDC indices
3. Download paired CT images and tumor segmentations
4. Train a tumor segmentation model with MONAI

## The nlst-seg Collection

We'll use the `nlst-seg` collection which contains:
- CT images from the National Lung Screening Trial (NLST)
- Manual expert annotations of lung nodules/tumors
- High-quality ground truth for tumor segmentation

## Setup environment

In [ ]:
!pip install -q monai idc-index pydicom itkwasm-dicom

## Setup imports

In [ ]:
import os
import tempfile

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from idc_index import IDCClient

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceLoss
import monai

monai.config.print_config()

## 1. Understanding Image-Derived Data in IDC

Beyond imaging modalities (CT, MR, PET), IDC contains **image-derived DICOM objects**:

| Modality | Name | Description |
|----------|------|-------------|
| **SEG** | Segmentation | Voxel-level labels (best for AI training) |
| **RTSTRUCT** | RT Structure Set | Contour-based annotations |
| **SR** | Structured Report | Measurements and findings |

Let's explore what's available.

In [ ]:
# Initialize client
client = IDCClient()

# What image-derived data exists in IDC?
derived_data = client.sql_query("""
    SELECT 
        Modality,
        COUNT(DISTINCT SeriesInstanceUID) as series_count,
        COUNT(DISTINCT collection_id) as collections
    FROM index
    WHERE Modality IN ('SEG', 'RTSTRUCT', 'SR')
    GROUP BY Modality
    ORDER BY series_count DESC
""")

print("Image-Derived DICOM Objects in IDC:")
print(derived_data.to_string(index=False))

## 2. Finding Manual Tumor Annotations

To understand where annotations come from, we can use:
- `source_DOI` - Links to publications describing the data
- `seg_index` - Detailed metadata including `AlgorithmType` (MANUAL vs AUTOMATIC)
- `collections_index` - Collection-level information

In [ ]:
# Fetch seg_index for segmentation metadata
client.fetch_index("seg_index")

# Find collections with MANUAL segmentations
manual_segs = client.sql_query("""
    SELECT 
        seg_info.collection_id,
        s.AlgorithmType,
        COUNT(DISTINCT s.SeriesInstanceUID) as seg_count,
        COUNT(DISTINCT seg_info.PatientID) as patients
    FROM seg_index s
    JOIN index seg_info ON s.SeriesInstanceUID = seg_info.SeriesInstanceUID
    WHERE s.AlgorithmType = 'MANUAL'
    GROUP BY seg_info.collection_id, s.AlgorithmType
    ORDER BY seg_count DESC
    LIMIT 10
""")

print("Collections with Manual Segmentations:")
print(manual_segs.to_string(index=False))

In [ ]:
# Look at nlst-seg collection specifically
nlst_seg_info = client.sql_query("""
    SELECT 
        seg_info.collection_id,
        seg_info.source_DOI,
        s.AlgorithmType,
        COUNT(DISTINCT s.SeriesInstanceUID) as seg_count,
        COUNT(DISTINCT seg_info.PatientID) as patients
    FROM seg_index s
    JOIN index seg_info ON s.SeriesInstanceUID = seg_info.SeriesInstanceUID
    WHERE seg_info.collection_id = 'nlst_seg'
    GROUP BY seg_info.collection_id, seg_info.source_DOI, s.AlgorithmType
""")

print("\nnlst_seg Collection:")
print(nlst_seg_info.to_string(index=False))
if len(nlst_seg_info) > 0 and nlst_seg_info.iloc[0]['source_DOI']:
    print(f"\nSource DOI for citation: {nlst_seg_info.iloc[0]['source_DOI']}")

## 3. Query Paired CT Images and Tumor Segmentations

The `seg_index.segmented_SeriesInstanceUID` links each segmentation to its source image.

In [ ]:
# Find CT images with their corresponding manual tumor segmentations
paired_data = client.sql_query("""
    SELECT
        src.SeriesInstanceUID as image_series_uid,
        seg.SeriesInstanceUID as seg_series_uid,
        src.PatientID,
        src.StudyDescription,
        seg.total_segments,
        ROUND(src.series_size_MB, 2) as image_size_mb
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    JOIN index seg_info ON seg.SeriesInstanceUID = seg_info.SeriesInstanceUID
    WHERE seg_info.collection_id = 'nlst_seg'
        AND src.Modality = 'CT'
        AND seg.AlgorithmType = 'MANUAL'
    ORDER BY src.series_size_MB ASC
    LIMIT 10
""")

print(f"Found {len(paired_data)} CT-Segmentation pairs:")
print(paired_data[['PatientID', 'total_segments', 'image_size_mb']].to_string(index=False))

## 4. Download the Data

In [ ]:
# Create download directory
data_dir = tempfile.mkdtemp(prefix="nlst_tumor_")
print(f"Download directory: {data_dir}")

# Use first 5 pairs for this demo
demo_data = paired_data.head(5)

# Collect all UIDs
all_uids = list(demo_data['image_series_uid']) + list(demo_data['seg_series_uid'])

print(f"\nDownloading {len(all_uids)} series ({len(demo_data)} image-seg pairs)...")

client.download_from_selection(
    seriesInstanceUID=all_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"
)

print("Download complete!")

## 5. Prepare Data for MONAI

In [ ]:
# Build data dictionaries
data_dicts = []
for _, row in demo_data.iterrows():
    data_dicts.append({
        "image": os.path.join(data_dir, row['image_series_uid']),
        "label": os.path.join(data_dir, row['seg_series_uid']),
        "patient_id": row['PatientID'],
    })

print(f"Created {len(data_dicts)} data dictionaries")
print(f"\nExample paths:")
print(f"  Image: {data_dicts[0]['image'][:60]}...")
print(f"  Label: {data_dicts[0]['label'][:60]}...")

In [ ]:
# Split into train/val
train_files = data_dicts[:4]
val_files = data_dicts[4:]

print(f"Training: {len(train_files)} cases")
print(f"Validation: {len(val_files)} cases")

## 6. Define MONAI Transforms

In [ ]:
from idc_monai.transforms import LoadDicomSegd

# Training transforms with augmentation
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadDicomSegd(keys=["label"]),  # Use specialized DICOM SEG loader
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    # Lung CT window
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-1000,  # Air
        a_max=400,    # Soft tissue
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    # Random patches centered on tumor
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1,  # Positive samples (with tumor)
        neg=1,  # Negative samples (background)
        num_samples=2,
    ),
    # Data augmentation
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
])

# Validation transforms (no augmentation)
val_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadDicomSegd(keys=["label"]),  # Use specialized DICOM SEG loader
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-1000,
        a_max=400,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
])

## 7. Create Datasets and Load a Sample

In [ ]:
# Create datasets
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

# Load a sample to inspect
sample = train_ds[0]

print(f"Image shape: {sample['image'].shape}")
print(f"Label shape: {sample['label'].shape}")
print(f"Unique labels: {torch.unique(sample['label']).numpy()}")

## 8. Visualize Tumor Annotations

DICOM SEG files contain color information for each segment in `recommendedDisplayRGBValue`. We extract this from the metadata to render segments with their DICOM-defined colors.

In [ ]:
def build_seg_colormap(overlay_info):
    """Build a matplotlib colormap from DICOM SEG segment colors.
    
    Args:
        overlay_info: Dictionary from LoadDicomSegd containing segmentAttributes
        
    Returns:
        ListedColormap with colors for each segment label
    """
    segment_attrs = overlay_info.get("segmentAttributes", [[]])
    
    # Flatten segment attributes (may be nested in groups)
    all_segments = []
    for group in segment_attrs:
        all_segments.extend(group)
    
    if not all_segments:
        return plt.cm.Reds  # Fallback for tumor visualization
    
    max_label = max(seg.get("labelID", 0) for seg in all_segments)
    
    # Build RGBA color array: index 0 = background (transparent)
    colors = np.zeros((max_label + 1, 4))
    colors[0] = [0, 0, 0, 0]  # Background transparent
    
    for seg in all_segments:
        label_id = seg.get("labelID", 0)
        rgb = seg.get("recommendedDisplayRGBValue", [255, 0, 0])  # Default red for tumors
        colors[label_id] = [rgb[0]/255, rgb[1]/255, rgb[2]/255, 1.0]
    
    return ListedColormap(colors)

# Load a validation sample for visualization (no random cropping)
vis_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadDicomSegd(keys=["label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
])
vis_sample = Dataset(data=data_dicts[:1], transform=vis_transforms)[0]

# Get data for visualization
image = vis_sample['image'][0].numpy()
label = vis_sample['label'][0].numpy()

# Build colormap from DICOM SEG metadata
overlay_info = vis_sample.get('label_meta_dict', {}).get('overlay_info', {})
seg_cmap = build_seg_colormap(overlay_info)

# Find slices containing tumor
tumor_slices = np.where(label.sum(axis=(0, 1)) > 0)[0]
print(f"Tumor present in {len(tumor_slices)} slices")

if len(tumor_slices) > 0:
    # Show middle slice with tumor
    z = tumor_slices[len(tumor_slices) // 2]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # CT image
    axes[0].imshow(image[:, :, z].T, cmap='gray', origin='lower', vmin=-1000, vmax=400)
    axes[0].set_title('CT Image')
    axes[0].axis('off')
    
    # Tumor annotation (using DICOM SEG colors)
    axes[1].imshow(label[:, :, z].T, cmap=seg_cmap, origin='lower',
                   vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
    axes[1].set_title('Manual Tumor Annotation\n(DICOM SEG colors)')
    axes[1].axis('off')
    
    # Overlay (using DICOM SEG colors)
    axes[2].imshow(image[:, :, z].T, cmap='gray', origin='lower', vmin=-1000, vmax=400)
    tumor_mask = np.ma.masked_where(label[:, :, z] == 0, label[:, :, z])
    axes[2].imshow(tumor_mask.T, cmap=seg_cmap, alpha=0.6, origin='lower',
                   vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
    axes[2].set_title('Overlay\n(DICOM SEG colors)')
    axes[2].axis('off')
    
    plt.suptitle(f'Manual Tumor Annotation (Slice {z})', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No tumor in this sample - try another")

## 9. Train Tumor Segmentation Model

In [ ]:
# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Create model (binary: tumor vs background)
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,  # Background + Tumor
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# Loss and optimizer
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop (abbreviated for demo)
max_epochs = 3
epoch_losses = []

# Convert multi-class labels to binary (any tumor = 1)
def to_binary(label):
    return (label > 0).float()

for epoch in range(max_epochs):
    print(f"\nEpoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch in train_loader:
        step += 1
        inputs = batch["image"].to(device)
        labels = to_binary(batch["label"]).to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"  Step {step}: Loss = {loss.item():.4f}")
    
    epoch_loss /= step
    epoch_losses.append(epoch_loss)
    print(f"  Epoch {epoch + 1} Mean Loss: {epoch_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, 'b-o')
plt.xlabel('Epoch')
plt.ylabel('Dice Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## 10. Generate Citation for the Data

In [ ]:
# Always cite the data sources!
citations = client.citations_from_selection(collection_id="nlst_seg")

print("Citations for nlst_seg data:")
for i, cite in enumerate(citations):
    print(f"\n{i+1}. {cite}")

## 11. Cleanup

In [ ]:
# Optionally cleanup
# import shutil
# shutil.rmtree(data_dir)

print(f"Data saved at: {data_dir}")

## Summary

In this tutorial, you learned how to:

1. **Discover image-derived data** (SEG, RTSTRUCT, SR) in IDC
2. **Find manual annotations** using `seg_index` with `AlgorithmType = 'MANUAL'`
3. **Link segmentations to source images** via `segmented_SeriesInstanceUID`
4. **Download paired data** from IDC
5. **Train a tumor segmentation model** with MONAI
6. **Generate citations** for proper attribution

## Key Queries

```sql
-- Find manual segmentations
SELECT * FROM seg_index WHERE AlgorithmType = 'MANUAL'

-- Link segmentation to source image
SELECT src.*, seg.SeriesInstanceUID as seg_uid
FROM seg_index seg
JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID

-- Get provenance
SELECT DISTINCT collection_id, source_DOI FROM index WHERE Modality = 'SEG'
```

## Next Steps

- Use `CacheDataset` for faster training on larger datasets
- Explore TotalSegmentator for 104 anatomical structures
- Add validation metrics and sliding window inference

## Resources

- [IDC Data Organization](https://learn.canceridc.dev/data/organization-of-data)
- [MONAI Segmentation Tutorials](https://github.com/Project-MONAI/tutorials/tree/main/3d_segmentation)